In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import matplotlib.patches as patches

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
coco_path = '../Data/Raw/training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json'
images_path = '../Data/Raw/training_data/quadrant-enumeration-disease/xrays'

with open(coco_path) as f:
    coco_file = json.load(f)

In [37]:
coco_file['images'][2]

{'height': 1316, 'width': 2909, 'id': 3, 'file_name': 'train_435.png'}

In [51]:
coco_file['annotations'][10]

{'iscrowd': 0,
 'image_id': 1,
 'bbox': [1405.1948051948052,
  724.6753246753246,
  75.32467532467535,
  353.2467532467533],
 'segmentation': [[1418,
   724,
   1476,
   729,
   1466,
   841,
   1477,
   941,
   1480,
   1075,
   1419,
   1077,
   1409,
   931,
   1405,
   790]],
 'id': 11,
 'area': 22286,
 'category_id_1': 2,
 'category_id_2': 1,
 'category_id_3': 2}

In [6]:
len(coco_file['images'])

705

In [ ]:
len(coco_file['annotations'])

3529

In [41]:
diagnosis_map = {
    0: "impacted",
    1: "caries",
    2: "periapical",
    3: "deep_caries"
}

quad_map = {
    0: "Upper Right",
    1: "Upper Left",
    2: "Lower Left",
    3: "Lower Right"
}

f_name = []
bbox = []
seg = []
quad = []
tooth_num = []
disease = []

for item in coco_file['images']:
    for i,ann in enumerate(coco_file['annotations']):
        if int(ann['image_id']) == int(item['id']):
            f_name.append(item['file_name'])
            bbox.append(ann['bbox'])
            seg.extend(ann['segmentation'])
            quad.append(quad_map[ann['category_id_1']])
            tooth_num.append(ann['category_id_2'])
            disease.append(diagnosis_map[ann['category_id_3']])

df = pd.DataFrame({'File_Name':f_name,
                   'bbox':bbox,
                   'Seg':seg,
                   'Quad':quad,
                   'Tooth_Num':tooth_num,
                   'Disease_Name':disease})

df

,File_Name,bbox,Seg,Quad,Tooth_Num,Disease_Name
0,train_673.png,"[542.0, 698.0, 220.0, 271.0]","[621, 703, 573, 744, 542, 885, 580, 945, 650, ...",Lower Right,7,impacted
1,train_673.png,"[1952.0, 693.0, 177.0, 270.0]","[2045, 693, 2109, 734, 2129, 915, 2047, 963, 2...",Lower Left,7,impacted
2,train_673.png,"[675.0, 708.0, 243.0, 300.0]","[784, 711, 754, 746, 737, 821, 678, 916, 675, ...",Lower Right,6,caries
3,train_673.png,"[1463.0, 725.0, 98.0, 425.0]","[1464, 749, 1513, 725, 1550, 760, 1555, 798, 1...",Lower Left,2,caries
4,train_673.png,"[1536.0, 753.0, 103.0, 381.0]","[1543, 796, 1590, 753, 1622, 796, 1629, 840, 1...",Lower Left,3,caries
...,...,...,...,...,...,...
3524,train_370.png,"[1851.2857142857142, 474.2857142857142, 117.14...","[1885, 477, 1868, 597, 1859, 657, 1851, 728, 1...",Upper Left,5,caries
3525,train_370.png,"[1959.0, 479.9999999999999, 127.0, 274.2857142...","[2005, 488, 1974, 479, 1965, 522, 1965, 588, 1...",Upper Left,6,caries
3526,train_370.png,"[2024.9999999999998, 463.0, 147.00000000000023...","[2064, 463, 2024, 471, 2036, 559, 2056, 628, 2...",Upper Left,7,deep_caries
3527,train_370.png,"[1921.7142857142856, 749.0, 221.28571428571445...","[1942, 783, 1924, 806, 1921, 835, 1944, 872, 1...",Lower Left,6,caries


In [42]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3529 entries, 0 to 3528
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   File_Name     3529 non-null   str   
 1   bbox          3529 non-null   object
 2   Seg           3529 non-null   object
 3   Quad          3529 non-null   str   
 4   Tooth_Num     3529 non-null   int64 
 5   Disease_Name  3529 non-null   str   
dtypes: int64(1), object(2), str(3)
memory usage: 271.3+ KB


In [44]:
df.isna().sum()

File_Name       0
bbox            0
Seg             0
Quad            0
Tooth_Num       0
Disease_Name    0
dtype: int64

In [43]:
df['File_Name'].value_counts()

File_Name
train_273.png    23
train_3.png      21
train_391.png    17
train_462.png    16
train_67.png     16
                 ..
train_52.png      1
train_111.png     1
train_33.png      1
train_389.png     1
train_238.png     1
Name: count, Length: 678, dtype: int64

In [45]:
df['File_Name'].drop_duplicates()

0       train_673.png
13      train_283.png
17       train_95.png
31      train_475.png
33       train_62.png
            ...      
3504    train_338.png
3509    train_657.png
3511    train_344.png
3514    train_599.png
3521    train_370.png
Name: File_Name, Length: 678, dtype: str